# radsum-qlora — QLoRA fine-tune of Qwen2-VL-2B for radiology summarization

End-to-end: install → data (IU X-ray / OpenI) → QLoRA train (~8–12h on T4) → clinical eval (CheXbert F1 + finding-hallucination rate) → baselines.
Budget for the whole notebook: ~14 GPU-hours (one Kaggle week).

In [ ]:
# 0. GPU check
import torch, subprocess
print(torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout)

In [ ]:
# 1. Dependencies (run once)
%pip install -q transformers peft bitsandbytes accelerate qwen-vl-utils datasets huggingface_hub pandas pyarrow pillow scikit-learn pyyaml tqdm

In [ ]:
# 2. Code (single source of truth = the GitHub repo)
!git clone https://github.com/habib-analyst/radsum-qlora 2>/dev/null || (cd radsum-qlora && git pull)
%cd radsum-qlora
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
import sys; sys.path.insert(0, 'src')

In [ ]:
# 3. Data: IU X-ray mirror (~2.1 GB) + CheXbert weights (~1.3 GB)
!mkdir -p data/raw weights
!python scripts/download_openi.py --out data/raw
!curl -sL --retry 3 -o weights/chexbert.pth https://huggingface.co/StanfordAIMI/RRG_scorers/resolve/main/chexbert.pth
!ls -la weights/chexbert.pth

In [ ]:
# 4. Build manifest + study-disjoint splits
!python -m radsum.data_build --raw data/raw --out data/processed
# point configs at the Kaggle paths
!sed -i 's|data/processed/manifest.parquet|/kaggle/working/radsum-qlora/data/processed/manifest.parquet|' configs/train.yaml configs/eval.yaml
!sed -i 's|weights/chexbert.pth|/kaggle/working/radsum-qlora/weights/chexbert.pth|' configs/eval.yaml

In [ ]:
# 5. TRAIN — QLoRA, 3 epochs (~8-12h on T4). Logs stream to train_log.jsonl
!python -m radsum.train --config configs/train.yaml --out_dir results/qlora_run 2>&1 | tee results/train_console.log

In [ ]:
# 6. EVAL main model: generate test impressions + CheXbert F1 + FHR
!python -m radsum.evaluate --config configs/eval.yaml --split test --adapter results/qlora_run/adapter_final --out results/eval_qlora.json

In [ ]:
# 7. BASELINES (each ~1-3h)
# 7a. zero-shot base 2B
!python -m radsum.evaluate --config configs/eval.yaml --split test --out results/eval_zeroshot2b.json
# 7b. text-only ablation: fine-tuned model, blank image at inference
#     (proves the image matters; expect FHR to rise)
!python -m radsum.evaluate --config configs/eval.yaml --split test --adapter results/qlora_run/adapter_final --blank_image --out results/eval_textonly.json

In [ ]:
# 7c. zero-shot 7B (bigger zero-shot reference; 4-bit fits on T4, ~15 GB download)
#     subset of 200 to bound eval time (~2h)
!sed 's|Qwen/Qwen2-VL-2B-Instruct|Qwen/Qwen2-VL-7B-Instruct|' configs/eval.yaml > configs/eval_7b.yaml
!python -m radsum.evaluate --config configs/eval_7b.yaml --split test --max_samples 200 --out results/eval_zeroshot7b.json

In [ ]:
# 8. Validate the FHR metric (perturbation / negation / agreement)
!python -m radsum.validate_metric --weights weights/chexbert.pth --manifest data/processed/manifest.parquet --n 200 --out results/metric_validation.json

In [ ]:
# 9. Summary table
import json, glob
for f in sorted(glob.glob('results/eval_*.json')):
    r = json.load(open(f))
    c = r['config']
    print(f"\n== {f} ==")
    print('  micro F1 (14):', round(r['chexbert_f1_all14']['micro_f1'], 4))
    print('  macro F1 (14):', round(r['chexbert_f1_all14']['macro_f1'], 4))
    print('  micro F1 (key5):', round(r['chexbert_f1_key5']['micro_f1'], 4))
    print('  FHR macro:', round(r['finding_hallucination_rate']['macro_fhr'], 4))
    print('  ROUGE-L:', round(r['rouge_l_f1_mean'], 4))
print('\nmetric validation:')
print(open('results/metric_validation.json').read())